# Telecom Customer Lifecycle & Revenue Analytics

## Synthetic Data Generation

This notebook generates a synthetic dataset for a fictional North American telecommunications company, Pinecrest Digital Services.

The dataset is designed to simulate customer, subscription, usage, billing, support, and acquisition data for the purpose of analyzing:

- Customer growth
- Customer churn
- Revenue and ARPU
- Customer acquisition cost (CAC)
- Customer lifetime value (LTV)
- Customer behavior and service usage
- Customer support and retention

The data is intentionally generated with realistic relationships between customer behavior and business outcomes so that the resulting analysis can support meaningful business questions.

In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Making results reproducible
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Project Parameters

The synthetic dataset represents Pinecrest Digital Services' customer base from January 2022 through December 2025.

The primary analysis period will be January through December 2025, while earlier customer activity is included to provide realistic customer tenure and lifetime history.

In [2]:
# Dataset parameters
NUM_CUSTOMERS = 15_000

CUSTOMER_START_DATE = "2022-01-01"
CUSTOMER_END_DATE = "2025-12-31"

ANALYSIS_START_DATE = "2025-01-01"
ANALYSIS_END_DATE = "2025-12-31"

## Markets

Pinecrest Digital Services operates in five North American markets:

- Ontario
- Quebec
- New York
- Michigan
- Ohio

Ontario and Quebec represent the company's largest Canadian markets, while the remaining markets represent its US operations.

In [3]:
# Assigning proportion of customers to the 5 markets
markets = {
    "Ontario": {
        "country": "Canada",
        "weight": 0.35
    },
    "Quebec": {
        "country": "Canada",
        "weight": 0.25
    },
    "New York": {
        "country": "United States",
        "weight": 0.15
    },
    "Michigan": {
        "country": "United States",
        "weight": 0.125
    },
    "Ohio": {
        "country": "United States",
        "weight": 0.125
    }
}

# Sanity check, sum of weights should be 1.0
print(sum(market["weight"] for market in markets.values()))

1.0


## Customer Segments

The customer base consists primarily of residential consumers, with a smaller small-business segment.

In [4]:
customer_segments = {
    "Consumer": 0.85,
    "Small Business": 0.15
}

# Sanity check
print(sum(customer_segments.values()))

1.0


## Customer Generation

Each customer receives:

- A unique customer ID
- A signup date
- A date of birth
- A country and market
- An acquisition channel
- A customer segment

Signup dates are distributed between 2022 and 2025 so that customers have different tenure lengths.

In [5]:
# Defining acquisition channels
acquisition_channels = {
    "Digital Advertising": 0.25,
    "Organic Search": 0.20,
    "Referral": 0.15,
    "Retail Store": 0.15,
    "Partner": 0.15,
    "Email Campaign": 0.10
}

# Sanity check
print(sum(acquisition_channels.values()))

1.0


In [6]:
# Generating customers
customers = pd.DataFrame({
    "customer_id": [
        f"C{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)
    ],
    "signup_date": pd.to_datetime(
        np.random.choice(
            pd.date_range(
                CUSTOMER_START_DATE,
                CUSTOMER_END_DATE
            ),
            size = NUM_CUSTOMERS
        )
    ),
    "market": np.random.choice(
        list(markets.keys()),
        size = NUM_CUSTOMERS,
        p = [m["weight"] for m in markets.values()]
    ),

    "customer_segment": np.random.choice(
        list(customer_segments.keys()),
        size = NUM_CUSTOMERS,
        p = list(customer_segments.values())
    ),

    "acquisition_channel": np.random.choice(
        list(acquisition_channels.keys()),
        size = NUM_CUSTOMERS,
        p = list(acquisition_channels.values())
    )
})

print(customers.head())

  customer_id signup_date    market customer_segment  acquisition_channel
0      C00001  2025-01-31   Ontario         Consumer  Digital Advertising
1      C00002  2025-12-30   Ontario         Consumer  Digital Advertising
2      C00003  2024-05-10   Ontario         Consumer       Organic Search
3      C00004  2025-07-18   Ontario         Consumer  Digital Advertising
4      C00005  2025-02-04  New York         Consumer             Referral


In [7]:
# Adding country column based on market
customers["country"] = customers["market"].map(
    lambda x:markets[x]["country"]
)

print(customers.head())

  customer_id signup_date    market customer_segment  acquisition_channel  \
0      C00001  2025-01-31   Ontario         Consumer  Digital Advertising   
1      C00002  2025-12-30   Ontario         Consumer  Digital Advertising   
2      C00003  2024-05-10   Ontario         Consumer       Organic Search   
3      C00004  2025-07-18   Ontario         Consumer  Digital Advertising   
4      C00005  2025-02-04  New York         Consumer             Referral   

         country  
0         Canada  
1         Canada  
2         Canada  
3         Canada  
4  United States  


In [8]:
# Generating DOB for age based analysis
reference_date = pd.Timestamp("2025-01-01")

ages = np.random.randint(
    18,
    75,
    size = NUM_CUSTOMERS
)

customers["date_of_birth"] = (
    reference_date - pd.to_timedelta(ages * 365.25, unit = "D")
)

print(customers.head())

  customer_id signup_date    market customer_segment  acquisition_channel  \
0      C00001  2025-01-31   Ontario         Consumer  Digital Advertising   
1      C00002  2025-12-30   Ontario         Consumer  Digital Advertising   
2      C00003  2024-05-10   Ontario         Consumer       Organic Search   
3      C00004  2025-07-18   Ontario         Consumer  Digital Advertising   
4      C00005  2025-02-04  New York         Consumer             Referral   

         country       date_of_birth  
0         Canada 1967-01-01 12:00:00  
1         Canada 2004-01-01 18:00:00  
2         Canada 1956-01-01 18:00:00  
3         Canada 1971-01-01 12:00:00  
4  United States 2001-01-01 00:00:00  


In [9]:
# Cleaning up the date columns
customers["signup_date"] = customers["signup_date"].dt.normalize()
customers["date_of_birth"] = customers["date_of_birth"].dt.normalize()

print(customers.head())

  customer_id signup_date    market customer_segment  acquisition_channel  \
0      C00001  2025-01-31   Ontario         Consumer  Digital Advertising   
1      C00002  2025-12-30   Ontario         Consumer  Digital Advertising   
2      C00003  2024-05-10   Ontario         Consumer       Organic Search   
3      C00004  2025-07-18   Ontario         Consumer  Digital Advertising   
4      C00005  2025-02-04  New York         Consumer             Referral   

         country date_of_birth  
0         Canada    1967-01-01  
1         Canada    2004-01-01  
2         Canada    1956-01-01  
3         Canada    1971-01-01  
4  United States    2001-01-01  


In [10]:
# Reordering columns for ease of use
customers = customers[
    [
        "customer_id",
        "signup_date",
        "date_of_birth",
        "country",
        "market",
        "customer_segment",
        "acquisition_channel"
    ]
]

print(customers.head())
print(customers.shape)

  customer_id signup_date date_of_birth        country    market  \
0      C00001  2025-01-31    1967-01-01         Canada   Ontario   
1      C00002  2025-12-30    2004-01-01         Canada   Ontario   
2      C00003  2024-05-10    1956-01-01         Canada   Ontario   
3      C00004  2025-07-18    1971-01-01         Canada   Ontario   
4      C00005  2025-02-04    2001-01-01  United States  New York   

  customer_segment  acquisition_channel  
0         Consumer  Digital Advertising  
1         Consumer  Digital Advertising  
2         Consumer       Organic Search  
3         Consumer  Digital Advertising  
4         Consumer             Referral  
(15000, 7)


## Data Validation

There should be zero missing values, all customer ids should be unique, and the market distribution should be reasonably close to the intended percentages

In [11]:
print(customers.isna().sum())

customer_id            0
signup_date            0
date_of_birth          0
country                0
market                 0
customer_segment       0
acquisition_channel    0
dtype: int64


In [12]:
print(customers["customer_id"].nunique())

15000


In [13]:
print(customers["market"].value_counts(normalize = True))

market
Ontario     0.354667
Quebec      0.248133
New York    0.150400
Ohio        0.127600
Michigan    0.119200
Name: proportion, dtype: float64


## Internet Plan Generation

Pinecrest Digital Services offers five broadband plans.

The plan structure is designed to create meaningful differences in:

- Monthly revenue
- Network usage
- Customer segment
- Contract type
- Customer lifetime value

In [14]:
# Generating plans
plans = pd.DataFrame({
    "plan_id": [
        "P001",
        "P002",
        "P003",
        "P004",
        "P005"
    ],
    "plan_name": [
        "Essential",
        "Standard",
        "Premium",
        "Ultra",
        "Business Pro"
    ],
    "monthly_price": [
        55,
        75,
        95,
        120,
        150
    ],
    "speed_mbps": [
        100,
        500,
        1000,
        1500,
        2000
    ],
    "contract_type": [
        "Month-to-month",
        "Month-to-month",
        "2-year",
        "2-year",
        "2-year"
    ]
})

print(plans)

  plan_id     plan_name  monthly_price  speed_mbps   contract_type
0    P001     Essential             55         100  Month-to-month
1    P002      Standard             75         500  Month-to-month
2    P003       Premium             95        1000          2-year
3    P004         Ultra            120        1500          2-year
4    P005  Business Pro            150        2000          2-year


## Subscription Generation

Each customer receives an initial broadband subscription.

Plan selection varies by customer segment to create realistic differences in customer value and usage. Subscription attributes will later be used to generate billing, usage, and churn behavior.

In [15]:
# Creating plan probabilities
consumer_plan_probabilities = {
    "Essential": 0.30,
    "Standard": 0.40,
    "Premium": 0.23,
    "Ultra": 0.07,
    "Business Pro": 0.00
}

business_plan_probabilities = {
    "Essential": 0.05,
    "Standard": 0.20,
    "Premium": 0.30,
    "Ultra": 0.25,
    "Business Pro": 0.20
}

# Sanity check
print(sum(consumer_plan_probabilities.values()))
print(sum(business_plan_probabilities.values()))

1.0
1.0


In [16]:
# Assigning initial plan based on segment
def assign_plan(segment):
    if segment == "Consumer":
        return np.random.choice(
            list(consumer_plan_probabilities.keys()),
            p = list(consumer_plan_probabilities.values())
        )
    else:
        return np.random.choice(
            list(business_plan_probabilities.keys()),
            p = list(business_plan_probabilities.values())
        )

customers["initial_plan"] = customers["customer_segment"].apply(assign_plan)

In [17]:
# Inspecting distribution
print(customers["initial_plan"].value_counts(normalize = True))

initial_plan
Standard        0.368133
Essential       0.265933
Premium         0.238000
Ultra           0.096000
Business Pro    0.031933
Name: proportion, dtype: float64


In [18]:
print(pd.crosstab(
    customers["customer_segment"],
    customers["initial_plan"],
    normalize = "index"
))

initial_plan      Business Pro  Essential   Premium  Standard     Ultra
customer_segment                                                       
Consumer              0.000000   0.303447  0.227624  0.395607  0.073321
Small Business        0.217037   0.048482  0.298142  0.208881  0.227458


In [19]:
# Setting subscription start dates to signup dates for now
subscriptions = customers[
    [
        "customer_id",
        "signup_date",
        "initial_plan",
        "customer_segment"
    ]
].copy()

subscriptions = subscriptions.rename(
    columns = {
        "signup_date": "start_date",
        "initial_plan": "plan_name"
    }
)

In [20]:
# Attaching plan details
subscriptions = subscriptions.merge(
    plans,
    on="plan_name",
    how="left"
)

In [21]:
print(subscriptions.columns)

Index(['customer_id', 'start_date', 'plan_name', 'customer_segment', 'plan_id',
       'monthly_price', 'speed_mbps', 'contract_type'],
      dtype='object')


In [22]:
print(subscriptions.head())

  customer_id start_date  plan_name customer_segment plan_id  monthly_price  \
0      C00001 2025-01-31      Ultra         Consumer    P004            120   
1      C00002 2025-12-30   Standard         Consumer    P002             75   
2      C00003 2024-05-10    Premium         Consumer    P003             95   
3      C00004 2025-07-18   Standard         Consumer    P002             75   
4      C00005 2025-02-04  Essential         Consumer    P001             55   

   speed_mbps   contract_type  
0        1500          2-year  
1         500  Month-to-month  
2        1000          2-year  
3         500  Month-to-month  
4         100  Month-to-month  


## Contract Type
For realism in later churn analysis, some customers should be on different contract arrangements. For example some higher tier customers may still be month-to-month

In [23]:
contract_override_probability = 0.10

In [24]:
def assign_contract(row):
    if row["contract_type"] == "2-year":
        if np.random.random() < contract_override_probability:
            return "Month-to-month"
        return row["contract_type"]
    return row["contract_type"]

In [25]:
subscriptions["contract_type"] = subscriptions.apply(
    assign_contract,
    axis = 1
)

In [26]:
pd.crosstab(
    subscriptions["plan_name"],
    subscriptions["contract_type"],
    dropna = False
)

contract_type,2-year,Month-to-month
plan_name,,
Business Pro,437,42
Essential,0,3989
Premium,3225,345
Standard,0,5522
Ultra,1311,129


## Discounts
Not every customer is going to be paying the same as others on the same plan. Some may have promotional discounts.

In [27]:
discount_options = [0, 0.10, 0.15, 0.20]
discount_probabilities = [0.45, 0.25, 0.20, 0.10]

In [28]:
# Adding discount column to subscriptions
subscriptions["discount_rate"] = np.random.choice(
    discount_options,
    size = len(subscriptions),
    p = discount_probabilities
)

In [29]:
# Adjusting monthly_price to reflect the discounts
subscriptions["monthly_price"] = (
    subscriptions["monthly_price"] * (1 - subscriptions["discount_rate"])
).round(2)

In [30]:
print(subscriptions.head(10))

  customer_id start_date  plan_name customer_segment plan_id  monthly_price  \
0      C00001 2025-01-31      Ultra         Consumer    P004          108.0   
1      C00002 2025-12-30   Standard         Consumer    P002           75.0   
2      C00003 2024-05-10    Premium         Consumer    P003           95.0   
3      C00004 2025-07-18   Standard         Consumer    P002           67.5   
4      C00005 2025-02-04  Essential         Consumer    P001           49.5   
5      C00006 2024-12-31      Ultra   Small Business    P004          120.0   
6      C00007 2024-11-10      Ultra         Consumer    P004           96.0   
7      C00008 2022-05-02      Ultra   Small Business    P004           96.0   
8      C00009 2023-04-12   Standard         Consumer    P002           75.0   
9      C00010 2025-05-23   Standard         Consumer    P002           67.5   

   speed_mbps   contract_type  discount_rate  
0        1500          2-year            0.1  
1         500  Month-to-month       

In [31]:
# Assigning every initial subscription a unique ID
subscriptions.insert(
    0,
    "subscription_id",
    [f"S{i:06d}" for i in range(1, len(subscriptions) + 1)]
)

In [32]:
print(subscriptions.head())

  subscription_id customer_id start_date  plan_name customer_segment plan_id  \
0         S000001      C00001 2025-01-31      Ultra         Consumer    P004   
1         S000002      C00002 2025-12-30   Standard         Consumer    P002   
2         S000003      C00003 2024-05-10    Premium         Consumer    P003   
3         S000004      C00004 2025-07-18   Standard         Consumer    P002   
4         S000005      C00005 2025-02-04  Essential         Consumer    P001   

   monthly_price  speed_mbps   contract_type  discount_rate  
0          108.0        1500          2-year            0.1  
1           75.0         500  Month-to-month            0.0  
2           95.0        1000          2-year            0.0  
3           67.5         500  Month-to-month            0.1  
4           49.5         100  Month-to-month            0.1  


## Subscription Status
At this point I will not randomly assign churn. I will generate that after establishing customer characteristics. For now I will set all subscriptions to "Active"

In [33]:
subscriptions["status"] = "Active"
subscriptions["end_date"] = pd.NaT

print(subscriptions.head())

  subscription_id customer_id start_date  plan_name customer_segment plan_id  \
0         S000001      C00001 2025-01-31      Ultra         Consumer    P004   
1         S000002      C00002 2025-12-30   Standard         Consumer    P002   
2         S000003      C00003 2024-05-10    Premium         Consumer    P003   
3         S000004      C00004 2025-07-18   Standard         Consumer    P002   
4         S000005      C00005 2025-02-04  Essential         Consumer    P001   

   monthly_price  speed_mbps   contract_type  discount_rate  status end_date  
0          108.0        1500          2-year            0.1  Active      NaT  
1           75.0         500  Month-to-month            0.0  Active      NaT  
2           95.0        1000          2-year            0.0  Active      NaT  
3           67.5         500  Month-to-month            0.1  Active      NaT  
4           49.5         100  Month-to-month            0.1  Active      NaT  


In [34]:
# Dropping customer_segment and speed_mbps since they are in other tables and reorganizing columns
subscriptions = subscriptions[
    [    
        "subscription_id",
        "customer_id",
        "plan_id",
        "plan_name",
        "start_date",
        "end_date",
        "status",
        "monthly_price",
        "discount_rate",
        "contract_type"
    ]
]

print(subscriptions.head())
print(subscriptions.shape)

  subscription_id customer_id plan_id  plan_name start_date end_date  status  \
0         S000001      C00001    P004      Ultra 2025-01-31      NaT  Active   
1         S000002      C00002    P002   Standard 2025-12-30      NaT  Active   
2         S000003      C00003    P003    Premium 2024-05-10      NaT  Active   
3         S000004      C00004    P002   Standard 2025-07-18      NaT  Active   
4         S000005      C00005    P001  Essential 2025-02-04      NaT  Active   

   monthly_price  discount_rate   contract_type  
0          108.0            0.1          2-year  
1           75.0            0.0  Month-to-month  
2           95.0            0.0          2-year  
3           67.5            0.1  Month-to-month  
4           49.5            0.1  Month-to-month  
(15000, 10)


In [35]:
#sanity checks
subscriptions.isna().sum()

subscription_id        0
customer_id            0
plan_id                0
plan_name              0
start_date             0
end_date           15000
status                 0
monthly_price          0
discount_rate          0
contract_type          0
dtype: int64

In [36]:
subscriptions["customer_id"].nunique()

15000

In [37]:
subscriptions["subscription_id"].nunique()

15000

In [38]:
subscriptions["plan_name"].value_counts()

plan_name
Standard        5522
Essential       3989
Premium         3570
Ultra           1440
Business Pro     479
Name: count, dtype: int64

In [39]:
pd.crosstab(
    customers["customer_segment"],
    subscriptions["plan_name"],
    normalize="index"
)

plan_name,Business Pro,Essential,Premium,Standard,Ultra
customer_segment,,,,,
Consumer,0.000000,0.303447,0.227624,0.395607,0.073321
Small Business,0.217037,0.048482,0.298142,0.208881,0.227458


## Generate Acquisition Data

Each customer is associated with an acquisition event representing how they were acquired.

Acquisition costs vary by channel because different marketing and sales channels require different levels of investment.

The acquisition data will later be used to calculate Customer Acquisition Cost (CAC) and compare acquisition efficiency across markets and channels.

In [40]:
# Defining average acquisition costs
acquisition_costs = {
    "Digital Advertising": 85,
    "Organic Search": 25,
    "Referral": 40,
    "Retail Store": 70,
    "Partner": 60,
    "Email Campaign": 20
}

In [41]:
# Generating acquisition dataframe
acquisitions = customers[
    [
        "customer_id",
        "signup_date",
        "market",
        "acquisition_channel"
    ]
].copy()

In [42]:
# Renaming date column
acquisitions = acquisitions.rename(
    columns = {
        "signup_date": "acquisition_date"
    }
)

In [43]:
# Generating actual acquisition costs using a normal distribution around the average
acquisitions["acquisition_cost"] = acquisitions[
    "acquisition_channel"
].apply(
    lambda channel: max(
        5,
        np.random.normal(
            acquisition_costs[channel],
            acquisition_costs[channel] * 0.20
        )
    )
).round(2)

In [44]:
print(acquisitions.head())

  customer_id acquisition_date    market  acquisition_channel  \
0      C00001       2025-01-31   Ontario  Digital Advertising   
1      C00002       2025-12-30   Ontario  Digital Advertising   
2      C00003       2024-05-10   Ontario       Organic Search   
3      C00004       2025-07-18   Ontario  Digital Advertising   
4      C00005       2025-02-04  New York             Referral   

   acquisition_cost  
0             87.78  
1             92.51  
2             31.91  
3             99.04  
4             44.25  


## Campaign Information
The acquisition channels would have different sub-categories so I will randomly assign relevant subcategories to each acquisition.

In [45]:
campaigns = {
    "Digital Advertising": [
        "Search Ads",
        "Social Media",
        "Display Ads"
    ],
    "Organic Search": [
        "SEO - Broadband",
        "SEO - Internet Plans"
    ],
    "Referral": [
        "Customer Referral Program"
    ],
    "Retail Store": [
        "Retail Promotion",
        "Storefront Campaign"
    ],
    "Partner": [
        "Retail Partner",
        "Corporate Partnership"
    ],
    "Email Campaign": [
        "New Customer Promotion",
        "Upgrade Promotion"
    ]
}

In [46]:
def assign_campaign(channel):
    return np.random.choice(campaigns[channel])

acquisitions["campaign"] = acquisitions[
    "acquisition_channel"
].apply(assign_campaign)

In [47]:
print(acquisitions.head())

  customer_id acquisition_date    market  acquisition_channel  \
0      C00001       2025-01-31   Ontario  Digital Advertising   
1      C00002       2025-12-30   Ontario  Digital Advertising   
2      C00003       2024-05-10   Ontario       Organic Search   
3      C00004       2025-07-18   Ontario  Digital Advertising   
4      C00005       2025-02-04  New York             Referral   

   acquisition_cost                   campaign  
0             87.78                 Search Ads  
1             92.51               Social Media  
2             31.91            SEO - Broadband  
3             99.04               Social Media  
4             44.25  Customer Referral Program  


In [48]:
# Assigning unique ID to each acquisition event
acquisitions.insert(
    0,
    "acquisition_id",
    [f"A{i:06d}" for i in range(1, len(acquisitions) + 1)]
)

In [49]:
# Organizing columns
acquisitions = acquisitions[
    [
        "acquisition_id",
        "customer_id",
        "acquisition_date",
        "market",
        "acquisition_channel",
        "campaign",
        "acquisition_cost"
    ]
]

print(acquisitions.head())

  acquisition_id customer_id acquisition_date    market  acquisition_channel  \
0        A000001      C00001       2025-01-31   Ontario  Digital Advertising   
1        A000002      C00002       2025-12-30   Ontario  Digital Advertising   
2        A000003      C00003       2024-05-10   Ontario       Organic Search   
3        A000004      C00004       2025-07-18   Ontario  Digital Advertising   
4        A000005      C00005       2025-02-04  New York             Referral   

                    campaign  acquisition_cost  
0                 Search Ads             87.78  
1               Social Media             92.51  
2            SEO - Broadband             31.91  
3               Social Media             99.04  
4  Customer Referral Program             44.25  


In [50]:
# Sanity checks
print(acquisitions.shape)

(15000, 7)


In [51]:
print(acquisitions.isna().sum())

acquisition_id         0
customer_id            0
acquisition_date       0
market                 0
acquisition_channel    0
campaign               0
acquisition_cost       0
dtype: int64


In [52]:
print(acquisitions["customer_id"].nunique())

15000


In [53]:
print(acquisitions["acquisition_channel"].value_counts())

acquisition_channel
Digital Advertising    3667
Organic Search         2971
Referral               2327
Partner                2304
Retail Store           2218
Email Campaign         1513
Name: count, dtype: int64


In [54]:
acquisitions.groupby(
    "acquisition_channel"
)["acquisition_cost"].agg(
    ["count", "mean", "min", "max"]
).round(2)

,count,mean,min,max
acquisition_channel,,,,
Digital Advertising,3667,85.32,21.74,143.83
Email Campaign,1513,19.85,5.27,33.99
Organic Search,2971,25.00,9.13,40.89
Partner,2304,60.11,21.18,99.84
Referral,2327,39.99,10.25,65.24
Retail Store,2218,70.49,20.26,113.59


## Monthly Usage Data

Monthly usage records are generated for each customer based on their subscription plan and customer behavior.

Usage patterns vary across customers and over time to support analysis of:

- Network utilization
- Customer usage behavior
- Plan utilization
- Changes in usage over time

In [55]:
# Creating a range of months from Jan-Dec 2025 to generate usage data for analysis
analysis_months = pd.date_range(
    start = ANALYSIS_START_DATE,
    end = ANALYSIS_END_DATE,
    freq = "MS"
)

print(analysis_months)

DatetimeIndex(['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01',
               '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01',
               '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01'],
              dtype='datetime64[ns]', freq='MS')


In [56]:
# Defining typical monthly usage by plan in GB per month
plan_usage_profiles = {
    "Essential": {
        "average_usage": 250,
        "std_dev": 75
    },
    "Standard": {
        "average_usage": 400,
        "std_dev": 100
    },
    "Premium": {
        "average_usage": 600,
        "std_dev": 150
    },
    "Ultra": {
        "average_usage": 850,
        "std_dev": 200
    },
    "Business Pro": {
        "average_usage": 1200,
        "std_dev": 300
    }
}

I will assign every customer a persistent usage factor so that generally heavy users remain heavier than average and light users remain lighter

In [57]:
customer_usage_factors = pd.DataFrame({
    "customer_id": customers["customer_id"],
    "usage_factor": np.random.normal(
        loc = 1.0,
        scale = 0.20,
        size = len(customers)
    )
})

In [58]:
# Preventing extreme values
customer_usage_factors["usage_factor"] = (
    customer_usage_factors["usage_factor"].clip(lower = 0.5, upper = 1.5)
)

In [59]:
print(customer_usage_factors.head(10))

  customer_id  usage_factor
0      C00001      0.927816
1      C00002      1.020036
2      C00003      0.901916
3      C00004      0.878120
4      C00005      1.071756
5      C00006      1.216008
6      C00007      0.883844
7      C00008      0.850189
8      C00009      0.832469
9      C00010      0.981851


Customers can't use their plans before they signed up, so the montly usage records will only include customers that had already signed up by that month

In [60]:
# Generating monthly customer records
usage_records = []

for month in analysis_months:

    active_customers = subscriptions[
        subscriptions["start_date"] <= month
    ].copy()

    active_customers = active_customers.merge(
        customer_usage_factors,
        on = "customer_id",
        how = "left"
    )

    for _, customer in active_customers.iterrows():

        profile = plan_usage_profiles[customer["plan_name"]]

        monthly_usage = np.random.normal(
            loc = profile["average_usage"] * customer["usage_factor"],
            scale = profile["std_dev"]
        )

        # Monthly usage cannot be negative
        monthly_usage = max(0, monthly_usage)

        peak_usage = monthly_usage * np.random.uniform(
            0.15,
            0.30
        )

        days_in_month = month.days_in_month

        average_daily_usage = (
            monthly_usage / days_in_month
        )

        usage_records.append({
            "customer_id": customer["customer_id"],
            "usage_month": month,
            "data_usage_gb": round(monthly_usage, 2),
            "peak_usage_gb": round(peak_usage, 2),
            "average_daily_usage_gb": round(average_daily_usage, 2)
        })

In [61]:
# Creating the usage dataframe
monthly_usage = pd.DataFrame(usage_records)
print(monthly_usage.head())

  customer_id usage_month  data_usage_gb  peak_usage_gb  \
0      C00003  2025-01-01         393.69         103.14   
1      C00006  2025-01-01         986.60         272.46   
2      C00007  2025-01-01         715.11         176.16   
3      C00008  2025-01-01         534.98         140.72   
4      C00009  2025-01-01         593.19         151.27   

   average_daily_usage_gb  
0                   12.70  
1                   31.83  
2                   23.07  
3                   17.26  
4                   19.14  


In [62]:
print(monthly_usage.shape)

(155728, 5)


In [63]:
# Sanity Checks
monthly_usage.isna().sum()

customer_id               0
usage_month               0
data_usage_gb             0
peak_usage_gb             0
average_daily_usage_gb    0
dtype: int64

In [64]:
#temporary merge for validation
usage_check = monthly_usage.merge(
    subscriptions[
        ["customer_id", "plan_name"]
    ],
    on = "customer_id",
    how = "left"
)

usage_check.groupby(
    "plan_name"
)["data_usage_gb"].mean().round(2)

plan_name
Business Pro    1183.13
Essential        250.80
Premium          602.46
Standard         399.90
Ultra            850.25
Name: data_usage_gb, dtype: float64

## Support Ticket Data

Support ticket records are generated to represent customer interactions with technical support and customer service.

Ticket volume and resolution outcomes will vary across customers, allowing for later analysis of operational performance, customer experience, and potential relationships with churn.

In [65]:
# Defining support ticket categories
ticket_categories = {
    "Technical Issue": 0.35,
    "Billing Issue": 0.20,
    "Service Outage": 0.15,
    "Account Inquiry": 0.15,
    "Plan Change": 0.10,
    "Cancellation Request": 0.05
}

# Sanity check
print(sum(ticket_categories.values()))

1.0


In [66]:
# Defining support ticket priority for generating resolution time later
priority_options = {
    "Low": 0.30,
    "Medium": 0.45,
    "High": 0.20,
    "Critical": 0.05
}

# Sanity check
print(sum(priority_options.values()))

1.0


In [67]:
# Using a Poisson distribution to decide how many tickets each customer has
customer_ticket_counts = pd.DataFrame({
    "customer_id": customers["customer_id"],
    "ticket_count": np.random.poisson(
        lam = 1.2,
        size = len(customers)
    )
})

print(customer_ticket_counts.head())

  customer_id  ticket_count
0      C00001             0
1      C00002             1
2      C00003             1
3      C00004             1
4      C00005             0


In [68]:
# checking distribution
print(customer_ticket_counts["ticket_count"].value_counts().sort_index())

ticket_count
0    4454
1    5512
2    3246
3    1316
4     344
5     103
6      23
7       2
Name: count, dtype: int64


In [69]:
# Generating ticket records
support_ticket_records = []

In [70]:
for _, customer in customer_ticket_counts.iterrows():

    if customer["ticket_count"] == 0:
        continue

    customer_id = customer["customer_id"]

    customer_signup_date = customers.loc[
        customers["customer_id"] == customer_id,
        "signup_date"
    ].iloc[0]

    possible_dates = pd.date_range(
        start = max(
            customer_signup_date,
            pd.Timestamp(ANALYSIS_START_DATE)
        ),
        end = pd.Timestamp(ANALYSIS_END_DATE),
        freq = "D"
    )

    for _ in range(customer["ticket_count"]):
        ticket_date = np.random.choice(possible_dates)

        category = np.random.choice(
            list(ticket_categories.keys()),
            p = list(ticket_categories.values())
        )

        priority = np.random.choice(
            list(priority_options.keys()),
            p = list(priority_options.values())
        )

        support_ticket_records.append({
            "customer_id": customer_id,
            "ticket_date": ticket_date,
            "ticket_category": category,
            "priority": priority
        })

In [71]:
support_tickets = pd.DataFrame(support_ticket_records)

print(support_tickets.head())

  customer_id ticket_date       ticket_category priority
0      C00002  2025-12-31       Technical Issue     High
1      C00003  2025-03-04  Cancellation Request   Medium
2      C00004  2025-09-22       Technical Issue     High
3      C00006  2025-10-20  Cancellation Request   Medium
4      C00007  2025-11-25         Billing Issue      Low


In [72]:
# Adding ticket IDs
support_tickets.insert(
    0,
    "ticket_id",
    [
        f"T{i:07d}" for i in range(1, len(support_tickets) + 1)
    ]
)

print(support_tickets.head())

  ticket_id customer_id ticket_date       ticket_category priority
0  T0000001      C00002  2025-12-31       Technical Issue     High
1  T0000002      C00003  2025-03-04  Cancellation Request   Medium
2  T0000003      C00004  2025-09-22       Technical Issue     High
3  T0000004      C00006  2025-10-20  Cancellation Request   Medium
4  T0000005      C00007  2025-11-25         Billing Issue      Low


Now I will generate resolution times. Higher-priority tickets will be more complex issues and generally take longer to resolve

In [73]:
# Defining average resolution time in hours
resolution_time_profiles = {
    "Low": 12,
    "Medium": 24,
    "High": 48,
    "Critical": 72
}

In [74]:
# Generating actual resolution times
support_tickets["resolution_time_hours"] = (
    support_tickets["priority"].apply(
        lambda priority: max(
            1,
            np.random.normal(
                resolution_time_profiles[priority],
                resolution_time_profiles[priority] * 0.30
            )
        )
    ).round()
)

print(support_tickets.head())

  ticket_id customer_id ticket_date       ticket_category priority  \
0  T0000001      C00002  2025-12-31       Technical Issue     High   
1  T0000002      C00003  2025-03-04  Cancellation Request   Medium   
2  T0000003      C00004  2025-09-22       Technical Issue     High   
3  T0000004      C00006  2025-10-20  Cancellation Request   Medium   
4  T0000005      C00007  2025-11-25         Billing Issue      Low   

   resolution_time_hours  
0                   74.0  
1                   32.0  
2                   39.0  
3                   26.0  
4                   12.0  


In [75]:
# Adding resolution status. Most should be resolved
support_tickets["resolution_status"] = np.random.choice(
    ["Resolved", "Unresolved"],
    size = len(support_tickets),
    p = [0.95, 0.05]
)

In [76]:
print(support_tickets.head())

  ticket_id customer_id ticket_date       ticket_category priority  \
0  T0000001      C00002  2025-12-31       Technical Issue     High   
1  T0000002      C00003  2025-03-04  Cancellation Request   Medium   
2  T0000003      C00004  2025-09-22       Technical Issue     High   
3  T0000004      C00006  2025-10-20  Cancellation Request   Medium   
4  T0000005      C00007  2025-11-25         Billing Issue      Low   

   resolution_time_hours resolution_status  
0                   74.0          Resolved  
1                   32.0          Resolved  
2                   39.0          Resolved  
3                   26.0        Unresolved  
4                   12.0          Resolved  


In [77]:
# Sanity checks
print(support_tickets.isna().sum())

ticket_id                0
customer_id              0
ticket_date              0
ticket_category          0
priority                 0
resolution_time_hours    0
resolution_status        0
dtype: int64


In [78]:
print(support_tickets.shape)

(17995, 7)


In [79]:
print(support_tickets["ticket_category"].value_counts(normalize = True))

ticket_category
Technical Issue         0.349430
Billing Issue           0.200722
Account Inquiry         0.146652
Service Outage          0.143873
Plan Change             0.105418
Cancellation Request    0.053904
Name: proportion, dtype: float64


In [80]:
print(support_tickets["resolution_status"].value_counts(normalize = True))

resolution_status
Resolved      0.950486
Unresolved    0.049514
Name: proportion, dtype: float64


In [81]:
support_tickets.groupby(
    "priority"
)["resolution_time_hours"].mean().round()

priority
Critical    72.0
High        48.0
Low         12.0
Medium      24.0
Name: resolution_time_hours, dtype: float64

## Monthly Billing Data

Monthly billing records represent the recurring revenue generated by each active customer.

Billing amounts are based on the customer's subscription plan, actual contract type, and applicable discount.

This dataset will support analysis of revenue, ARPU, customer lifetime value, and revenue trends.

In [82]:
# Creating billing records
billing_records = []

for month in analysis_months:
    active_customers = subscriptions[
        subscriptions["start_date"] <= month
    ].copy()

    for _, customer in active_customers.iterrows():

        billing_records.append({
            "customer_id": customer["customer_id"],
            "billing_month": month,
            "monthly_price": customer["monthly_price"]
        })

In [83]:
monthly_billing = pd.DataFrame(billing_records)

print(monthly_billing.head())

  customer_id billing_month  monthly_price
0      C00003    2025-01-01           95.0
1      C00006    2025-01-01          120.0
2      C00007    2025-01-01           96.0
3      C00008    2025-01-01           96.0
4      C00009    2025-01-01           75.0


In [84]:
# Adding billing ID
monthly_billing.insert(
    0,
    "billing_id",
    [
        f"B{i:07d}" for i in range(1, len(monthly_billing) + 1)
    ]
)

print(monthly_billing.head())

  billing_id customer_id billing_month  monthly_price
0   B0000001      C00003    2025-01-01           95.0
1   B0000002      C00006    2025-01-01          120.0
2   B0000003      C00007    2025-01-01           96.0
3   B0000004      C00008    2025-01-01           96.0
4   B0000005      C00009    2025-01-01           75.0


In [85]:
# Sanity checks
# Missing values
print(monthly_billing.isna().sum())

billing_id       0
customer_id      0
billing_month    0
monthly_price    0
dtype: int64


In [86]:
# Number of billing records
print(monthly_billing.shape)

(155728, 4)


In [87]:
# Billing Months
print(monthly_billing["billing_month"].value_counts().sort_index())

billing_month
2025-01-01    11208
2025-02-01    11565
2025-03-01    11878
2025-04-01    12183
2025-05-01    12510
2025-06-01    12832
2025-07-01    13127
2025-08-01    13464
2025-09-01    13776
2025-10-01    14090
2025-11-01    14405
2025-12-01    14690
Name: count, dtype: int64


In [88]:
# Verifying that billing uses the actual subscription price
billing_check = monthly_billing.merge(
    subscriptions[["customer_id", "monthly_price"]],
    on = "customer_id",
    how = "left",
    suffixes = ("_billed", "_subscription")
)

(
    billing_check["monthly_price_billed"]
    - billing_check["monthly_price_subscription"]
).abs().max()

np.float64(0.0)

In [89]:
# Revenue by month
monthly_billing.groupby(
    "billing_month"
)["monthly_price"].sum().round(2)

billing_month
2025-01-01     843095.75
2025-02-01     869627.50
2025-03-01     893288.25
2025-04-01     916267.00
2025-05-01     940776.75
2025-06-01     964784.25
2025-07-01     986465.75
2025-08-01    1011295.50
2025-09-01    1034531.50
2025-10-01    1058013.50
2025-11-01    1081475.25
2025-12-01    1102734.75
Name: monthly_price, dtype: float64

In [90]:
# Revenue by plan
billing_check = monthly_billing.merge(
    subscriptions[
        ["customer_id", "plan_name"]
    ],
    on = "customer_id",
    how = "left"
)

In [91]:
billing_check.groupby(
    "plan_name"
)["monthly_price"].mean().round(2)

plan_name
Business Pro    138.23
Essential        50.92
Premium          87.90
Standard         69.32
Ultra           110.82
Name: monthly_price, dtype: float64

## Customer Churn Data

Customer churn is generated using a combination of subscription characteristics and customer behavior.

Factors influencing churn risk include contract type, customer tenure, support activity, unresolved support issues, and acquisition channel.

The resulting churn data will support analysis of churn rate, retention, customer lifetime value, and drivers of customer attrition.

In [92]:
churn_data = subscriptions[
    [
        "customer_id",
        "start_date",
        "plan_name",
        "contract_type",
        "discount_rate",
    ]
].copy()

In [93]:
# Calculating customer tenure
analysis_end = pd.Timestamp(ANALYSIS_END_DATE)

churn_data["tenure_months"] = (
    (
        analysis_end.year - pd.to_datetime(
            churn_data["start_date"]
        ).dt.year
    ) * 12
    +
    (
        analysis_end.month - pd.to_datetime(
            churn_data["start_date"]
        ).dt.month
    )
).clip(lower = 0)

In [94]:
churn_data["tenure_months"].describe()

count    15000.000000
mean        23.212933
std         13.744402
min          0.000000
25%         11.000000
50%         23.000000
75%         35.000000
max         47.000000
Name: tenure_months, dtype: float64

In [95]:
# Adding support ticket activity
ticket_summary = support_tickets.groupby(
    "customer_id"
).agg(
    ticket_count = ("ticket_id", "count"),
    unresolved_tickets = (
        "resolution_status",
        lambda x: (x == "Unresolved").sum()
    )
).reset_index()

In [96]:
churn_data = churn_data.merge(
    ticket_summary,
    on = "customer_id",
    how = "left"
)

In [97]:
# Customers with no tickets will be NaN so replacing them with zeros
churn_data[
    ["ticket_count", "unresolved_tickets"]
] = churn_data[
    ["ticket_count", "unresolved_tickets"]
].fillna(0)

print(churn_data.head())

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  
0             11           0.0                 0.0  
1              0           1.0                 0.0  
2             19           1.0                 0.0  
3              5           1.0                 0.0  
4             10           0.0                 0.0  


In [98]:
# adding average usage
usage_summary = monthly_usage.groupby(
    "customer_id"
).agg(
    average_monthly_usage_gb = (
        "data_usage_gb",
        "mean"
    )
).reset_index()

In [99]:
churn_data = churn_data.merge(
    usage_summary,
    on = "customer_id",
    how = "left"
)

print(churn_data.head())

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  average_monthly_usage_gb  
0             11           0.0                 0.0                853.184545  
1              0           1.0                 0.0                       NaN  
2             19           1.0                 0.0                581.334167  
3              5           1.0                 0.0                368.934000  
4             10           0.0                 0.0                225.928000  


In [100]:
#filling NaN with zeroes
churn_data["average_monthly_usage_gb"] = churn_data[
    "average_monthly_usage_gb"
].fillna(0)

print(churn_data.head())

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  average_monthly_usage_gb  
0             11           0.0                 0.0                853.184545  
1              0           1.0                 0.0                  0.000000  
2             19           1.0                 0.0                581.334167  
3              5           1.0                 0.0                368.934000  
4             10           0.0                 0.0                225.928000  


In [101]:
# Adding acquisition channel
churn_data = churn_data.merge(
    acquisitions[
        [
            "customer_id",
            "acquisition_channel"
        ]
    ],
    on = "customer_id",
    how = "left"
)

### Churn Risk Score
I will now establish the relationships for the synthetic data to contain to churn

In [102]:
# Base probability
churn_data["churn_probability"] = 0.08

In [103]:
# Month-to-month customers are more likely to leave due to not having contract cancellation fees
churn_data.loc[
    churn_data["contract_type"] == "Month-to-month",
    "churn_probability"
] += 0.12

In [104]:
# Each additional support ticket increases customer dissatisfaction and thus churn
churn_data["churn_probability"] += (
    churn_data["ticket_count"] * 0.015
)

In [105]:
# Unresolved tickets have substantially stronger effects than resolved ones
churn_data["churn_probability"] += (
    churn_data["unresolved_tickets"] * 0.08
)

In [106]:
# Customers with less than 6 months of tenure will have additional churn risk due to not having brand loyalty yet
churn_data.loc[
    churn_data["tenure_months"] < 6,
    "churn_probability"
] += 0.05

In [107]:
# Acquisition channels will have slightly different churn rates to give more variance for analysis later
churn_data.loc[
    churn_data["acquisition_channel"] == "Digital Advertising",
    "churn_probability"
] += 0.02

churn_data.loc[
    churn_data["acquisition_channel"] == "Referral",
    "churn_probability"
] -= 0.02 

In [108]:
# Capping probabilities to reasonable range
churn_data["churn_probability"] = (
    churn_data["churn_probability"].clip(lower = 0.01, upper = 0.60)
)

print(churn_data.head())

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  average_monthly_usage_gb  \
0             11           0.0                 0.0                853.184545   
1              0           1.0                 0.0                  0.000000   
2             19           1.0                 0.0                581.334167   
3              5           1.0                 0.0                368.934000   
4             10           0.0                 0.0                225.928000   

   acquisition_channel  churn_probability  
0  Digital Advertising              0.100  
1  Digital

In [109]:
churn_data["churn_probability"].describe()

count    15000.000000
mean         0.191030
std          0.066767
min          0.060000
25%          0.125000
50%          0.210000
75%          0.230000
max          0.515000
Name: churn_probability, dtype: float64

### Churn Status
I will now randomly determine whether each customer churned based on their calculated probability

In [110]:
churn_data["churned"] = (
    np.random.random(len(churn_data)) < churn_data["churn_probability"]
)

# Converting to integer (0=retained, 1=churned)
churn_data["churned"] = (
    churn_data["churned"].astype(int)
)

print(churn_data.head())

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  average_monthly_usage_gb  \
0             11           0.0                 0.0                853.184545   
1              0           1.0                 0.0                  0.000000   
2             19           1.0                 0.0                581.334167   
3              5           1.0                 0.0                368.934000   
4             10           0.0                 0.0                225.928000   

   acquisition_channel  churn_probability  churned  
0  Digital Advertising              0.100    

In [111]:
print(churn_data["churned"].value_counts())

churned
0    12163
1     2837
Name: count, dtype: int64


In [112]:
print(churn_data["churned"].mean())

0.18913333333333332


### Churn Dates
Customers who churned should have a date between their signup date and the end of the analysis period

In [113]:
churn_data["churn_date"] = pd.NaT

In [114]:
for index, row in churn_data[
    churn_data["churned"] == 1
].iterrows():
    
    start = pd.Timestamp(row["start_date"])
    end = pd.Timestamp(ANALYSIS_END_DATE)

    if start < end:
        days_available = (end-start).days

        random_days = np.random.randint(
            1,
            days_available + 1
        )

        churn_data.loc[
            index,
            "churn_date",
        ] = start + pd.Timedelta(
            days = random_days
        )

In [115]:
print(churn_data.head(10))

  customer_id start_date  plan_name   contract_type  discount_rate  \
0      C00001 2025-01-31      Ultra          2-year            0.1   
1      C00002 2025-12-30   Standard  Month-to-month            0.0   
2      C00003 2024-05-10    Premium          2-year            0.0   
3      C00004 2025-07-18   Standard  Month-to-month            0.1   
4      C00005 2025-02-04  Essential  Month-to-month            0.1   
5      C00006 2024-12-31      Ultra          2-year            0.0   
6      C00007 2024-11-10      Ultra          2-year            0.2   
7      C00008 2022-05-02      Ultra          2-year            0.2   
8      C00009 2023-04-12   Standard  Month-to-month            0.0   
9      C00010 2025-05-23   Standard  Month-to-month            0.1   

   tenure_months  ticket_count  unresolved_tickets  average_monthly_usage_gb  \
0             11           0.0                 0.0                853.184545   
1              0           1.0                 0.0                  0

### Final Churn Table

In [116]:
# Dropping unneeded intermediate variables
customer_churn = churn_data[
    [
        "customer_id",
        "churned",
        "churn_date"
    ]
].copy()

print(customer_churn.head())

  customer_id  churned churn_date
0      C00001        0        NaT
1      C00002        0        NaT
2      C00003        0        NaT
3      C00004        0        NaT
4      C00005        0        NaT


In [117]:
# Sanity checks
customer_churn.shape

(15000, 3)

In [118]:
# Checking that churned customers have churn dates
customer_churn[
    customer_churn["churned"] == 1
]["churn_date"].isna().sum()

np.int64(0)

In [119]:
# Checking that retained customers do not have churn dates
customer_churn[
    customer_churn["churned"] == 0
]["churn_date"].notna().sum()

np.int64(0)

In [120]:
# Churn by contract check
churn_data.groupby(
    "contract_type"
)["churned"].mean().round(3)

contract_type
2-year            0.109
Month-to-month    0.229
Name: churned, dtype: float64

In [121]:
# Churn by support activity check
churn_data.groupby(
    "ticket_count"
)["churned"].mean().head(10).round(3)

ticket_count
0.0    0.168
1.0    0.186
2.0    0.207
3.0    0.191
4.0    0.276
5.0    0.291
6.0    0.478
7.0    0.000
Name: churned, dtype: float64

## Appling Churn to Monthly Data

Monthly usage and billing records are removed for periods after a customer's churn month.

The churn month itself is retained because the customer was active during that month.

In [122]:
# Creating a simplified churn lookup
churn_lookup = customer_churn[
    [
        "customer_id",
        "churn_date"
    ]
].copy()

In [123]:
# Applying churn to monthly usage
monthly_usage = monthly_usage.merge(
    churn_lookup,
    on = "customer_id",
    how = "left"
)

print(monthly_usage.head())

  customer_id usage_month  data_usage_gb  peak_usage_gb  \
0      C00003  2025-01-01         393.69         103.14   
1      C00006  2025-01-01         986.60         272.46   
2      C00007  2025-01-01         715.11         176.16   
3      C00008  2025-01-01         534.98         140.72   
4      C00009  2025-01-01         593.19         151.27   

   average_daily_usage_gb churn_date  
0                   12.70        NaT  
1                   31.83        NaT  
2                   23.07        NaT  
3                   17.26        NaT  
4                   19.14        NaT  


In [124]:
# Filtering out post-churn usage
monthly_usage = monthly_usage[
    monthly_usage["churn_date"].isna()
    |
    (
        monthly_usage["usage_month"].dt.to_period("M")
        <= monthly_usage["churn_date"].dt.to_period("M")
    )
].copy()

In [125]:
# Removing temporary churn date
monthly_usage = monthly_usage.drop(
    columns = "churn_date"
)

In [126]:
# Applying churn to billing
monthly_billing = monthly_billing.merge(
    churn_lookup,
    on = "customer_id",
    how = "left"
)

print(monthly_billing.head())

  billing_id customer_id billing_month  monthly_price churn_date
0   B0000001      C00003    2025-01-01           95.0        NaT
1   B0000002      C00006    2025-01-01          120.0        NaT
2   B0000003      C00007    2025-01-01           96.0        NaT
3   B0000004      C00008    2025-01-01           96.0        NaT
4   B0000005      C00009    2025-01-01           75.0        NaT


In [127]:
# Filtering out post-churn billing records
monthly_billing = monthly_billing[
    monthly_billing["churn_date"].isna()
    |
    (
        monthly_billing["billing_month"].dt.to_period("M")
        <= monthly_billing["churn_date"].dt.to_period("M")
    )
].copy()

In [128]:
# Removing temporary churn date
monthly_billing = monthly_billing.drop(
    columns = "churn_date"
)

In [129]:
# Sanity Checks
# Checking to make sure no post-churn usage records
usage_check = monthly_usage.merge(
    churn_lookup,
    on = "customer_id",
    how = "left"
)

In [130]:
post_churn_usage = usage_check[
    usage_check["churn_date"].notna()
    &
    (
        usage_check["usage_month"].dt.to_period("M")
        >
        usage_check["churn_date"].dt.to_period("M")
    )
]

print(len(post_churn_usage))

0


In [131]:
# Checking to make sure no post-churn billing records
billing_check = monthly_billing.merge(
    churn_lookup,
    on = "customer_id",
    how = "left"
)

In [132]:
post_churn_billing = billing_check[
    billing_check["churn_date"].notna()
    &
    (
        billing_check["billing_month"].dt.to_period("M")
        >
        billing_check["churn_date"].dt.to_period("M")
    )
]

print(len(post_churn_billing))

0


In [133]:
# Checking that churn reduced data
print("Usage records:", len(monthly_usage))
print("Billing records:", len(monthly_billing))

Usage records: 135668
Billing records: 135668


In [134]:
example_customer = customer_churn[
    customer_churn["churned"] == 1
].iloc[0]

print(example_customer)

customer_id                 C00012
churned                          1
churn_date     2025-08-27 00:00:00
Name: 11, dtype: object


In [135]:
monthly_usage[
    monthly_usage["customer_id"] == example_customer["customer_id"]
]

,customer_id,usage_month,data_usage_gb,peak_usage_gb,average_daily_usage_gb
6,C00012,2025-01-01,222.17,47.57,7.17
11215,C00012,2025-02-01,236.09,38.48,8.43
22781,C00012,2025-03-01,477.96,74.37,15.42
34659,C00012,2025-04-01,355.87,57.39,11.86
46842,C00012,2025-05-01,475.64,73.83,15.34
59353,C00012,2025-06-01,445.59,69.28,14.85
72185,C00012,2025-07-01,321.16,95.92,10.36
85313,C00012,2025-08-01,353.27,62.52,11.40


In [136]:
monthly_billing[
    monthly_billing["customer_id"] == example_customer["customer_id"]
]

,billing_id,customer_id,billing_month,monthly_price
6,B0000007,C00012,2025-01-01,63.75
11215,B0011216,C00012,2025-02-01,63.75
22781,B0022782,C00012,2025-03-01,63.75
34659,B0034660,C00012,2025-04-01,63.75
46842,B0046843,C00012,2025-05-01,63.75
59353,B0059354,C00012,2025-06-01,63.75
72185,B0072186,C00012,2025-07-01,63.75
85313,B0085314,C00012,2025-08-01,63.75


## Network Events

Network events represent service outages or network incidents affecting customers across different markets.

These records will support analysis of operational efficiency, network reliability, customer impact, and potential relationships between service issues and churn.

In [137]:
# Defining event types
network_event_types = {
    "Service Outage": 0.40,
    "Network Degredation": 0.30,
    "Equipment Failure": 0.20,
    "Planned Maintenance": 0.10
}

# Sanity check
print(sum(network_event_types.values()))

1.0


In [138]:
# Defining severity levels
network_severity = {
    "Minor": 0.45,
    "Moderate": 0.35,
    "Major": 0.15,
    "Critical": 0.05
}

# Sanity check
print(sum(network_severity.values()))

1.0


In [139]:
analysis_months = pd.date_range(
    start=ANALYSIS_START_DATE,
    end=ANALYSIS_END_DATE,
    freq="MS"
)

In [140]:
# Generating approx 3-5 events per month

network_event_records = []

event_id = 1

for month in analysis_months:

    number_of_events = np.random.poisson(lam = 4)

    for _ in range(number_of_events):

        market = np.random.choice(
            customers["market"].unique()
        )

        event_type = np.random.choice(
            list(network_event_types.keys()),
            p = list(network_event_types.values())
        )

        severity = np.random.choice(
            list(network_severity.keys()),
            p = list(network_severity.values())
        )

        network_event_records.append({
            "event_id": f"N{event_id:05d}",
            "event_month": month,
            "market": market,
            "event_type": event_type,
            "severity": severity
        })

        event_id += 1

In [141]:
# Creating dataframe
network_events = pd.DataFrame(network_event_records)

In [142]:
print(network_events.head())

  event_id event_month    market         event_type  severity
0   N00001  2025-01-01    Quebec  Equipment Failure     Minor
1   N00002  2025-01-01    Quebec     Service Outage     Minor
2   N00003  2025-02-01    Quebec     Service Outage     Minor
3   N00004  2025-02-01    Quebec     Service Outage     Minor
4   N00005  2025-02-01  Michigan  Equipment Failure  Moderate


In [143]:
# Generating effected customers
affected_customer_profiles = {
    "Minor": (50, 150),
    "Moderate": (150, 500),
    "Major": (500, 1500),
    "Critical": (1500, 4000)
}

In [144]:
network_events["affected_customers"] = (
    network_events["severity"].apply(
        lambda severity: np.random.randint(
            affected_customer_profiles[severity][0],
            affected_customer_profiles[severity][1]
        )
    )
)

print(network_events.head())

  event_id event_month    market         event_type  severity  \
0   N00001  2025-01-01    Quebec  Equipment Failure     Minor   
1   N00002  2025-01-01    Quebec     Service Outage     Minor   
2   N00003  2025-02-01    Quebec     Service Outage     Minor   
3   N00004  2025-02-01    Quebec     Service Outage     Minor   
4   N00005  2025-02-01  Michigan  Equipment Failure  Moderate   

   affected_customers  
0                 111  
1                  59  
2                  53  
3                 110  
4                 347  


In [145]:
# Generating downtime
downtime_profiles = {
    "Minor": (1, 4),
    "Moderate": (2, 8),
    "Major": (4, 16),
    "Critical": (8, 36)
}

In [146]:
network_events["downtime_hours"] = (
    network_events["severity"].apply(
        lambda severity: np.random.uniform(
            downtime_profiles[severity][0],
            downtime_profiles[severity][1]
        )
    ).round()
)

In [147]:
print(network_events.head())

  event_id event_month    market         event_type  severity  \
0   N00001  2025-01-01    Quebec  Equipment Failure     Minor   
1   N00002  2025-01-01    Quebec     Service Outage     Minor   
2   N00003  2025-02-01    Quebec     Service Outage     Minor   
3   N00004  2025-02-01    Quebec     Service Outage     Minor   
4   N00005  2025-02-01  Michigan  Equipment Failure  Moderate   

   affected_customers  downtime_hours  
0                 111             3.0  
1                  59             3.0  
2                  53             1.0  
3                 110             4.0  
4                 347             4.0  


In [148]:
# Sanity checks
network_events.isna().sum()

event_id              0
event_month           0
market                0
event_type            0
severity              0
affected_customers    0
downtime_hours        0
dtype: int64

In [149]:
len(network_events)

52

In [150]:
network_events["market"].value_counts()

market
Ontario     13
Quebec      12
Ohio        12
Michigan     8
New York     7
Name: count, dtype: int64

In [151]:
network_events["severity"].value_counts(
    normalize = True
).round(3)

severity
Minor       0.519
Moderate    0.308
Major       0.135
Critical    0.038
Name: proportion, dtype: float64

In [152]:
network_events.groupby(
    "severity"
)["downtime_hours"].mean().round(1)

severity
Critical    18.0
Major        7.1
Minor        2.7
Moderate     5.1
Name: downtime_hours, dtype: float64

In [153]:
network_events.groupby(
    "severity"
)["affected_customers"].mean().round()

severity
Critical    3364.0
Major        983.0
Minor        100.0
Moderate     326.0
Name: affected_customers, dtype: float64

## Final Data Validation and Export

The generated datasets are validated for missing values, duplicate primary keys, and key relationships before being exported for analysis.

The final datasets will be used in the SQLite and Tableau analysis stages of the project.

In [154]:
# Creating a dictionary of datasets to run the same checks across all of them
datasets = {
    "customers": customers,
    "plans": plans,
    "subscriptions": subscriptions,
    "acquisitions": acquisitions,
    "monthly_usage": monthly_usage,
    "support_tickets": support_tickets,
    "monthly_billing": monthly_billing,
    "customer_churn": customer_churn,
    "network_events": network_events
}

In [155]:
# Missing values check
for name, dataset in datasets.items():
    print(f"\n{name}")
    print(dataset.isna().sum())


customers
customer_id            0
signup_date            0
date_of_birth          0
country                0
market                 0
customer_segment       0
acquisition_channel    0
initial_plan           0
dtype: int64

plans
plan_id          0
plan_name        0
monthly_price    0
speed_mbps       0
contract_type    0
dtype: int64

subscriptions
subscription_id        0
customer_id            0
plan_id                0
plan_name              0
start_date             0
end_date           15000
status                 0
monthly_price          0
discount_rate          0
contract_type          0
dtype: int64

acquisitions
acquisition_id         0
customer_id            0
acquisition_date       0
market                 0
acquisition_channel    0
campaign               0
acquisition_cost       0
dtype: int64

monthly_usage
customer_id               0
usage_month               0
data_usage_gb             0
peak_usage_gb             0
average_daily_usage_gb    0
dtype: int64

support_tick

In [156]:
# Duplicate id checks
print("Duplicate customer IDs:", customers["customer_id"].duplicated().sum())

print("Duplicate plan IDs:", plans["plan_id"].duplicated().sum())

print(
    "Duplicate ticket IDs:",
    support_tickets["ticket_id"].duplicated().sum()
)

print(
    "Duplicate billing IDs:",
    monthly_billing["billing_id"].duplicated().sum()
)

print(
    "Duplicate event IDs:",
    network_events["event_id"].duplicated().sum()
)

Duplicate customer IDs: 0
Duplicate plan IDs: 0
Duplicate ticket IDs: 0
Duplicate billing IDs: 0
Duplicate event IDs: 0


In [157]:
# Checking that every customer referenced in other datasets exists in customers
customer_ids = set(customers["customer_id"])

for name, dataset in {
    "subscriptions": subscriptions,
    "acquisitions": acquisitions,
    "monthly_usage": monthly_usage,
    "support_tickets": support_tickets,
    "monthly_billing": monthly_billing,
    "customer_churn": customer_churn
}.items():
    
    missing_customers = (
        set(dataset["customer_id"])
        - customer_ids
    )
    
    print(
        f"{name}: {len(missing_customers)} missing customer IDs"
    )

subscriptions: 0 missing customer IDs
acquisitions: 0 missing customer IDs
monthly_usage: 0 missing customer IDs
support_tickets: 0 missing customer IDs
monthly_billing: 0 missing customer IDs
customer_churn: 0 missing customer IDs


In [158]:
# Checking final dataset sizes
for name, dataset in datasets.items():
    print(
        f"{name}: "
        f"{dataset.shape[0]:,} rows, "
        f"{dataset.shape[1]} columns"
    )

customers: 15,000 rows, 8 columns
plans: 5 rows, 5 columns
subscriptions: 15,000 rows, 10 columns
acquisitions: 15,000 rows, 7 columns
monthly_usage: 135,668 rows, 5 columns
support_tickets: 17,995 rows, 7 columns
monthly_billing: 135,668 rows, 4 columns
customer_churn: 15,000 rows, 3 columns
network_events: 52 rows, 7 columns


## Dataset Export

In [159]:
# Creating folder for raw data if it doesn't exist
data_path = Path("data")

In [160]:
data_path.mkdir(
    parents = True,
    exist_ok = True
)

for name, dataset in datasets.items():
    dataset.to_csv(
        data_path / f"{name}.csv",
        index = False
    )

In [161]:
# Confirming file creation
list(data_path.iterdir())

[WindowsPath('data/acquisitions.csv'),
 WindowsPath('data/customers.csv'),
 WindowsPath('data/customer_churn.csv'),
 WindowsPath('data/monthly_billing.csv'),
 WindowsPath('data/monthly_usage.csv'),
 WindowsPath('data/network_events.csv'),
 WindowsPath('data/plans.csv'),
 WindowsPath('data/subscriptions.csv'),
 WindowsPath('data/support_tickets.csv')]